<a href="https://colab.research.google.com/github/Nikita-Sudarshan/flyrank-ml-starter/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import subprocess

if not os.path.exists("flyrank-ml-starter"):
    !git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git

print("Current directory:", os.getcwd())
print("Repository exists:", os.path.exists("flyrank-ml-starter"))

Cloning into 'flyrank-ml-starter'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 224 (delta 108), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 2.60 MiB | 9.83 MiB/s, done.
Resolving deltas: 100% (108/108), done.
Current directory: /content
Repository exists: True


In [2]:
import pandas as pd

df = pd.read_csv(
    "flyrank-ml-starter/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# ### Honest grouped validation

# The Week-5 model was re-evaluated using a grouped train/test split based on `client_id`. The grouping prevents pages from the same client from appearing in both training and test sets.

# The split contained 23,837 training rows from 25 clients and 6,163 test rows from 7 clients. `client_id` was used only for grouping and was excluded from the model features. `trend_direction` was used as the target, while `trend_pct` was excluded because it is derived from the target proxy.

# On the grouped test set, the model achieved a Precision@20 of 1.000 compared with 0.250 for the simple baseline. The model also maintained Precision@100 of 1.000 and Precision@500 of 0.998, while Precision@1000 was 0.954.

# These results are observed performance on this dataset and split. They indicate that the model provides a stronger ranking of pages associated with the observed `down` label than the baseline under this validation design. They do not establish that refreshing a page will cause recovery or that the model predicts Google's ranking behavior.

In [4]:
# | Ranking depth | Baseline Precision | Model Precision |
# |---|---:|---:|
# | @20 | 0.250 | 1.000 |
# | @50 | 0.260 | 1.000 |
# | @100 | 0.230 | 1.000 |
# | @200 | 0.230 | 1.000 |
# | @500 | 0.260 | 0.998 |
# | @1000 | 0.303 | 0.954 |

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Target
y = df["trend_direction"].eq("down").astype(int)

# Groups — used only for splitting
groups = df["client_id"]

# Features — remove target, leakage-prone trend field, and client identity
X = df.drop(
    columns=[
        "trend_direction",
        "trend_pct",
        "client_id"
    ]
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["number"]).columns
categorical_features = X_train.select_dtypes(exclude=["number"]).columns

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

# Same model family as W05
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [7]:
# Check which original features are still present
print("Features used by the model:")
print(list(X.columns))

Features used by the model:
['content_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [8]:
# Check whether any remaining feature is strongly associated with the target
import pandas as pd

numeric_cols = X.select_dtypes(include="number").columns

correlations = (
    df[numeric_cols]
    .corrwith(y)
    .abs()
    .sort_values(ascending=False)
)

print("Top numeric correlations with the target:")
print(correlations.head(15))

Top numeric correlations with the target:
days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
impressions_last_30d      0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d           0.071935
sessions_last_30d         0.063842
ctr                       0.061911
clicks_90d                0.039680
engaged_sessions_90d      0.035402
avg_position              0.029035
clicks_prev_30d           0.028716
days_with_sessions        0.025055
dtype: float64


In [9]:
import numpy as np

# Predicted probability for every test example
y_prob = model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(y_prob))
print("Minimum probability:", round(y_prob.min(), 4))
print("Maximum probability:", round(y_prob.max(), 4))
print("Mean probability:", round(y_prob.mean(), 4))

print("\nProbability percentiles:")
for p in [50, 75, 90, 95, 99, 99.5, 99.9]:
    print(f"{p}th percentile:", round(np.percentile(y_prob, p), 4))

Number of test predictions: 6163
Minimum probability: 0.0
Maximum probability: 1.0
Mean probability: 0.5187

Probability percentiles:
50th percentile: 0.5563
75th percentile: 0.7087
90th percentile: 0.8508
95th percentile: 0.9711
99th percentile: 1.0
99.5th percentile: 1.0
99.9th percentile: 1.0


In [10]:
# Compare precision at different ranking depths

for k in [20, 50, 100, 200, 500, 1000]:
    top_indices = np.argsort(y_prob)[::-1][:k]
    precision = y_test.iloc[top_indices].mean()
    print(f"Precision@{k}: {precision:.4f}")

Precision@20: 1.0000
Precision@50: 1.0000
Precision@100: 1.0000
Precision@200: 1.0000
Precision@500: 0.9980
Precision@1000: 0.9540


In [11]:
import numpy as np

y_prob = model.predict_proba(X_test)[:, 1]

top_k = 20
top_indices = np.argsort(y_prob)[::-1][:top_k]

precision_at_20 = y_test.iloc[top_indices].mean()

print("Grouped-split Precision@20:", round(precision_at_20, 4))

Grouped-split Precision@20: 1.0


In [12]:
# Baseline: rank by a simple observed signal
# Use impressions_last_30d as the baseline ranking signal.

baseline_scores = X_test["impressions_last_30d"].fillna(0)

baseline_indices = np.argsort(baseline_scores.to_numpy())[::-1]

for k in [20, 50, 100, 200, 500, 1000]:
    top_indices = baseline_indices[:k]
    precision = y_test.iloc[top_indices].mean()
    print(f"Baseline Precision@{k}: {precision:.4f}")

Baseline Precision@20: 0.2500
Baseline Precision@50: 0.2600
Baseline Precision@100: 0.2300
Baseline Precision@200: 0.2300
Baseline Precision@500: 0.2600
Baseline Precision@1000: 0.3030


In [13]:
top_20 = df.iloc[test_idx].copy()

top_20["predicted_probability"] = y_prob

top_20 = top_20.sort_values(
    "predicted_probability",
    ascending=False
).head(20)

print(
    top_20[
        [
            "client_id",
            "trend_direction",
            "predicted_probability"
        ]
    ].to_string(index=False)
)

        client_id trend_direction  predicted_probability
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_f369cb89fc            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_f369cb89fc            down                    1.0
client_4e07408562            down                    1.0
client_4e07408562            down                    1.0
client_f369cb89fc            do

In [14]:
print("Features currently used by the model:\n")

for i, col in enumerate(X.columns, 1):
    print(f"{i}. {col}")

Features currently used by the model:

1. content_id
2. search_volume
3. competition
4. competition_level
5. cpc
6. content_type
7. main_intent
8. word_count
9. char_count
10. provider_used
11. model_used
12. impressions_90d
13. clicks_90d
14. pageviews_90d
15. sessions_90d
16. users_90d
17. engaged_sessions_90d
18. ai_sessions_90d
19. scroll_events_90d
20. days_with_impressions
21. days_with_sessions
22. impressions_last_30d
23. clicks_last_30d
24. sessions_last_30d
25. impressions_prev_30d
26. clicks_prev_30d
27. sessions_prev_30d
28. content_age_days
29. age_tier
30. age_tier_order
31. days_since_last_update
32. freshness_tier
33. word_count_tier
34. char_count_tier
35. ctr
36. avg_position
37. engagement_rate
38. scroll_rate
39. ai_traffic_pct
40. impression_tier
41. position_tier


In [15]:
print("Unique trend_direction values:")
print(df["trend_direction"].value_counts())

print("\nTrend-related columns:")
print([
    col for col in df.columns
    if "trend" in col.lower()
])

Unique trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend-related columns:
['trend_direction', 'trend_pct']


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
# Final feature-set leakage audit

target_column = "trend_direction"
excluded_columns = ["trend_pct", "client_id"]

print("Target:")
print(f"- {target_column}")

print("\nExplicitly excluded:")
for col in excluded_columns:
    print(f"- {col}")

print("\nFinal model features:")
print(f"- Total features: {len(X.columns)}")

print("\nChecking for target or excluded columns in features:")

leakage_columns = [
    col for col in X.columns
    if col in [target_column] + excluded_columns
]

if len(leakage_columns) == 0:
    print("PASS — target and explicitly excluded leakage-risk columns are not model features.")
else:
    print("FAIL — these columns are still present:")
    print(leakage_columns)

Target:
- trend_direction

Explicitly excluded:
- trend_pct
- client_id

Final model features:
- Total features: 41

Checking for target or excluded columns in features:
PASS — target and explicitly excluded leakage-risk columns are not model features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [17]:
# ### Original claim

# The model predicts which webpages will decline and identifies pages that should be refreshed.

# ### Safer research claim

# Under the grouped client-level validation split, the model showed higher observed Precision@20 than the simple baseline when ranking webpages associated with the observed `down` trend label. The results are directional and can support a prioritized review queue for content teams, but they do not prove that a refresh will improve performance or predict Google's ranking decisions.

In [18]:
# ### W06 self-check

# - [x] Two methodology findings/questions addressed
# - [x] Model evaluated under a grouped client-level split
# - [x] Baseline and model evaluated on the same test set
# - [x] Target and leakage-risk columns audited
# - [x] `trend_pct` excluded from model features
# - [x] `client_id` used only for grouping and excluded from model features
# - [x] Claims use observed, directional, and decision-support language
# - [x] No causal claim about refresh impact
# - [x] No claim about Google's ranking algorithm

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.